# Draft Recommendation Playground

Interactive tool for testing the MCTS draft agent with full parameter control.

**Quick start:**
1. Run all cells (`Kernel → Restart & Run All`)
2. Select map/mode and P1/P2 side
3. Add any already-picked brawlers and bans
4. Tune MCTS parameters (especially **UCB1 C** — see below)
5. Click **▶ Run MCTS**

---

### How picks are ranked

Picks are sorted by **visit count** — how many MCTS simulations were routed through that child node. This is the canonical AlphaZero metric, not the Q-value.

**Why visit count, not win prob (Q-value)?**  
UCB1 exploits high-value nodes: if a brawler consistently yields good rollouts, the tree directs more simulations there. So visit count reflects *sustained* performance across many samples, while Q-value (`value_sum / visit_count`) reflects the *mean* win probability from those samples. With few visits, Q-values have high variance — a brawler that got 20 visits and happened to hit three strong rollouts shows a high Q but is unreliable. Visit count is more robust because it's the accumulated judgment of the search itself.

**Can they disagree?** Yes — especially at high UCB1 C (exploration mode) or low sim counts. When they diverge, trust visit count. The ⚠ / ⚡ flags indicate when visit counts are too low to trust either metric.

---

### UCB1 C guide (exploration constant)
| C value | Effect | Use when |
|---------|--------|----------|
| 0.5 (default) | Aggressive exploitation — most visits go to pick #1 | You only care about the best pick |
| 1.5–2.0 | Balanced — picks 1–5 get meaningful visit counts | You want reliable top-5 |
| 4.0–8.0 | Broad exploration — visits spread across 10+ brawlers | You want trustworthy top-10 |

Higher C reduces confidence in pick #1 but dramatically improves reliability of picks #2–10.

---

**Complete draft:** if you enter all 3 picks for both teams, the tool skips MCTS and directly evaluates the terminal win probability using the FM model.

In [ ]:
import sys
from pathlib import Path

_src = (Path(".").resolve().parent / "src").resolve()

import numpy as np
import ipywidgets as w
from IPython.display import display, HTML, clear_output

from bsdraft.mcts.recommend import recommend, RecommendResult
from bsdraft.mcts.evaluator import FMEvaluator
from bsdraft.fm.model import FMInference
from bsdraft.data.matchup_db import MatchupDB
from bsdraft.selfplay.generate import load_map_mode_pairs

try:
    from bsdraft.data.prep import SEASON_CONFIGS
except ImportError:
    SEASON_CONFIGS = {
        "s42": {"data_dir": "../season42", "db_path": "../data/season42.db"},
        "s48": {"data_dir": "../season48", "db_path": "../data/season48.db"},
    }

print("Imports OK")


In [ ]:
# ── Season selection ──────────────────────────────────────────────────────────
SEASON = "s49"   # change to "s42" if needed

cfg      = SEASON_CONFIGS[SEASON]
DATA_DIR = Path(cfg["data_dir"])
DB_PATH  = Path(cfg["db_path"])

evaluator = FMEvaluator(FMInference.load(DATA_DIR / "fm_model.pkl"))
db        = MatchupDB.load(DATA_DIR / "matchup_db.pkl")
vocab     = evaluator._fm.schema.vocab
pairs     = load_map_mode_pairs(DB_PATH)

# Optional: load trained policy (if self-play training has been run)
policy = None
try:
    from bsdraft.selfplay.policy_net import PolicyInference
    _policy_path = DATA_DIR / "policy" / "policy_best.pkl"
    if _policy_path.exists():
        policy = PolicyInference.load(_policy_path)
        print(f"Loaded trained policy: {_policy_path}")
    else:
        print("No trained policy found — using counter-rate prior only.")
        print(f"  (expected: {_policy_path})")
except Exception as e:
    print(f"Could not load policy: {e}")

# Build map/mode lookups
all_maps = sorted(set(m for m, _ in pairs))
modes_by_map: dict[str, list[str]] = {}
for m, mode in sorted(set(pairs)):
    modes_by_map.setdefault(m, []).append(mode)

brawler_list = sorted(vocab)
print(f"\nSeason {SEASON}: {len(brawler_list)} brawlers, {len(set(pairs))} map/mode pairs")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Interactive Draft Playground Widget
# ═══════════════════════════════════════════════════════════════════════════════

from bsdraft.mcts.state import DraftState   # for complete-draft evaluation

# ── BrawlerPicker: search + add/remove UI ─────────────────────────────────────

class BrawlerPicker:
    """Search input + filtered list + confirmed picks panel."""

    def __init__(self, label: str, max_items: int = 3, color: str = "#2980b9"):
        self.max_items = max_items
        self._items: list[str] = []

        self.search = w.Text(
            placeholder="Search…",
            layout=w.Layout(width="190px"),
        )
        self.dropdown = w.Select(
            options=brawler_list,
            layout=w.Layout(width="190px", height="110px"),
        )
        self.add_btn = w.Button(
            description="+ Add",
            button_style="success",
            layout=w.Layout(width="80px"),
        )
        self.remove_btn = w.Button(
            description="− Remove",
            button_style="danger",
            layout=w.Layout(width="90px"),
        )
        self.current = w.Select(
            options=[],
            layout=w.Layout(width="190px", height="80px"),
        )
        self.status = w.HTML(value=f"<span style='color:gray;font-size:0.8em;'>0 / {max_items}</span>")

        self.search.observe(self._filter, names="value")
        self.add_btn.on_click(self._add)
        self.remove_btn.on_click(self._remove)

        self.widget = w.VBox([
            w.HTML(f"<b style='color:{color};'>{label}</b>"),
            self.search,
            self.dropdown,
            w.HBox([self.add_btn, self.remove_btn]),
            w.HBox([w.HTML("<span style='font-size:0.85em;color:gray;'>Confirmed:</span>"), self.status]),
            self.current,
        ], layout=w.Layout(margin="0 16px 0 0"))

    def _filter(self, change):
        q = change["new"].strip().upper()
        self.dropdown.options = [b for b in brawler_list if q in b] if q else brawler_list

    def _add(self, _):
        sel = self.dropdown.value
        if sel and sel not in self._items and len(self._items) < self.max_items:
            self._items.append(sel)
            self._refresh()

    def _remove(self, _):
        sel = self.current.value
        if sel in self._items:
            self._items.remove(sel)
            self._refresh()

    def _refresh(self):
        self.current.options = self._items[:]
        self.status.value = f"<span style='color:gray;font-size:0.8em;'>{len(self._items)} / {self.max_items}</span>"

    @property
    def picks(self) -> list[str]:
        return self._items[:]

    def clear(self):
        self._items = []
        self._refresh()


# ── Draft state controls ──────────────────────────────────────────────────────

_init_map  = all_maps[0]
_init_mode = modes_by_map[_init_map][0]

map_dropdown = w.Dropdown(
    options=all_maps,
    value=_init_map,
    layout=w.Layout(width="300px"),
)
mode_dropdown = w.Dropdown(
    options=modes_by_map[_init_map],
    value=_init_mode,
    layout=w.Layout(width="200px"),
)

def _on_map_change(change):
    modes = modes_by_map.get(change["new"], [])
    mode_dropdown.options = modes
    mode_dropdown.value   = modes[0] if modes else None

map_dropdown.observe(_on_map_change, names="value")

p1p2_toggle = w.ToggleButtons(
    options=[("P1 — First pick", True), ("P2 — Second pick", False)],
    value=True,
    layout=w.Layout(width="360px"),
)
skill_slider = w.FloatSlider(
    value=2.0, min=0.5, max=4.5, step=0.1,
    description="Skill ns:",
    style={"description_width": "70px"},
    layout=w.Layout(width="360px"),
    readout_format=".1f",
)
skill_note = w.HTML(
    "<span style='font-size:0.8em;color:gray;'>"
    "1.0=Bronze · 1.5=Gold · 2.0=Mythic · 3.0=Legendary · 4.0=Elite"
    "</span>"
)

# ── MCTS parameter controls ───────────────────────────────────────────────────

sims_slider = w.IntSlider(
    value=10_000, min=500, max=100_000, step=500,
    description="Sims:",
    style={"description_width": "70px"},
    layout=w.Layout(width="380px"),
)
ucb1c_slider = w.FloatLogSlider(
    value=2.0, base=10, min=-0.3, max=1.0, step=0.05,
    description="UCB1 C:",
    style={"description_width": "70px"},
    layout=w.Layout(width="380px"),
    readout_format=".2f",
)
ucb1c_note = w.HTML(
    "<span style='font-size:0.8em;color:gray;'>"
    "0.5 = exploit (best pick #1) &nbsp;·&nbsp; 2.0 = balanced &nbsp;·&nbsp; 5+ = explore (better picks #2–10)"
    "</span>"
)
ntop_slider = w.IntSlider(
    value=10, min=3, max=20, step=1,
    description="Show top:",
    style={"description_width": "70px"},
    layout=w.Layout(width="380px"),
)
min_pr_slider = w.FloatSlider(
    value=0.005, min=0.0, max=0.025, step=0.001,
    description="Min pick%:",
    style={"description_width": "70px"},
    layout=w.Layout(width="380px"),
    readout_format=".3f",
)
min_pr_note = w.HTML(
    "<span style='font-size:0.8em;color:gray;'>"
    "Filter out rarely-picked brawlers from the tree (0 = no filter)"
    "</span>"
)
puct_slider = w.FloatSlider(
    value=0.7, min=0.0, max=1.0, step=0.05,
    description="PUCT α:",
    style={"description_width": "70px"},
    layout=w.Layout(width="380px"),
    readout_format=".2f",
)
puct_note = w.HTML(
    "<span style='font-size:0.8em;color:gray;'>"
    "Prior blend at opp-turn nodes (0=pick-rate only, 1=counter-rate only)"
    "</span>"
)
use_policy_chk = w.Checkbox(
    value=(policy is not None),
    description="Use trained policy prior",
    disabled=(policy is None),
)
seed_input = w.IntText(
    value=42,
    description="RNG seed:",
    style={"description_width": "70px"},
    layout=w.Layout(width="180px"),
)

# ── Brawler pickers ───────────────────────────────────────────────────────────

my_picker  = BrawlerPicker("My Picks",  max_items=3, color="#27ae60")
opp_picker = BrawlerPicker("Opp Picks", max_items=3, color="#e74c3c")
ban_picker = BrawlerPicker("Bans",      max_items=6, color="#7f8c8d")

# ── Run / Clear / Reset buttons + output ─────────────────────────────────────

run_btn = w.Button(
    description="▶  Run MCTS",
    button_style="primary",
    layout=w.Layout(width="160px", height="38px"),
)
clear_btn = w.Button(
    description="Clear output",
    layout=w.Layout(width="110px", height="38px"),
)
reset_btn = w.Button(
    description="Reset draft",
    button_style="warning",
    layout=w.Layout(width="110px", height="38px"),
)
output = w.Output()

# ── HTML formatters ───────────────────────────────────────────────────────────

def _visit_bar(frac: float, max_px: int = 120) -> str:
    px = max(1, int(frac * max_px))
    if frac > 0.15:
        color = "#27ae60"
    elif frac > 0.05:
        color = "#f39c12"
    elif frac > 0.02:
        color = "#3498db"
    else:
        color = "#bdc3c7"
    return (
        f'<div style="display:inline-block;vertical-align:middle;'
        f'background:#ecf0f1;border-radius:3px;width:{max_px}px;height:8px;">'
        f'<div style="background:{color};width:{px}px;height:8px;border-radius:3px;"></div>'
        f'</div>'
    )

def _format_complete_draft(state: DraftState, win_prob: float) -> str:
    """HTML card for a complete draft — shows FM win probability for each team."""
    my_win  = win_prob
    opp_win = 1.0 - win_prob
    p1p2    = "P1" if state.is_first_pick else "P2"
    bans_s  = f" | bans: {', '.join(sorted(state.bans))}" if state.bans else ""

    # Colour the dominant team
    if my_win > 0.55:
        my_color, opp_color = "#27ae60", "#e74c3c"
        verdict = "My team favoured"
    elif opp_win > 0.55:
        my_color, opp_color = "#e74c3c", "#27ae60"
        verdict = "Opponent favoured"
    else:
        my_color = opp_color = "#f39c12"
        verdict = "Even matchup"

    def _prob_bar(p: float, color: str, width: int = 200) -> str:
        px = int(p * width)
        return (
            f'<div style="display:inline-block;background:#ecf0f1;border-radius:4px;'
            f'width:{width}px;height:14px;vertical-align:middle;">'
            f'<div style="background:{color};width:{px}px;height:14px;border-radius:4px;"></div>'
            f'</div>'
        )

    my_s  = ", ".join(sorted(state.my_team))
    opp_s = ", ".join(sorted(state.opp_team))
    return f"""
    <div style="font-family:monospace;max-width:640px;margin:12px 0;">
      <div style="background:#2c3e50;color:#ecf0f1;padding:10px 14px;border-radius:6px 6px 0 0;">
        <b>Draft Complete — FM Win Probability</b>
        &nbsp;|&nbsp;{state.mode}&nbsp;/&nbsp;{state.map_name}&nbsp;|&nbsp;{p1p2}{bans_s}
      </div>
      <div style="background:white;padding:18px 20px;border:1px solid #ddd;">
        <div style="margin-bottom:14px;">
          <span style="color:#27ae60;font-weight:bold;">My team:</span>
          <span style="margin-left:8px;">{my_s}</span>
        </div>
        <div style="margin-bottom:6px;">
          {_prob_bar(my_win, my_color)}
          &nbsp;<b style="color:{my_color};font-size:1.2em;">{my_win*100:.1f}%</b>
          &nbsp;<span style="color:gray;font-size:0.85em;">my win probability</span>
        </div>
        <div style="margin-bottom:14px;">
          {_prob_bar(opp_win, opp_color)}
          &nbsp;<b style="color:{opp_color};font-size:1.2em;">{opp_win*100:.1f}%</b>
          &nbsp;<span style="color:gray;font-size:0.85em;">opponent win probability</span>
        </div>
        <div style="margin-bottom:6px;">
          <span style="color:#e74c3c;font-weight:bold;">Opp team:</span>
          <span style="margin-left:8px;">{opp_s}</span>
        </div>
      </div>
      <div style="background:#ecf0f1;padding:6px 14px;font-size:0.82em;border-radius:0 0 6px 6px;">
        <b>{verdict}</b> &nbsp;·&nbsp;
        FM model direct evaluation (no MCTS needed — draft is complete)
        &nbsp;·&nbsp; skill_ns={state.skill_ns:.1f}
      </div>
    </div>
    """

def _format_result(result: RecommendResult, ucb1_c: float) -> str:
    l2      = result.layer2
    l1      = result.layer1
    state   = result.state
    picks   = l2["top_picks"]
    total   = l2["total_simulations"]
    n_avail = l2["n_available"]
    root_q  = l2["root_q"]
    label   = l2["confidence_label"]
    turn    = "My pick" if result.whose_turn == "mine" else "Opponent predicted pick"
    p1p2    = "P1" if state.is_first_pick else "P2"
    bans_s  = f" | bans: {', '.join(sorted(state.bans))}" if state.bans else ""

    label_colors = {
        "Dominant pick": "#e74c3c",
        "Strong pick": "#e67e22",
        "Solid pick": "#27ae60",
        "Even matchup — multiple viable options": "#2980b9",
    }
    lc    = label_colors.get(label, "#7f8c8d")
    my_s  = ", ".join(sorted(state.my_team))  or "—"
    opp_s = ", ".join(sorted(state.opp_team)) or "—"

    html = f"""
    <div style="font-family:monospace;max-width:820px;margin:12px 0;">
      <div style="background:#2c3e50;color:#ecf0f1;padding:10px 14px;border-radius:6px 6px 0 0;">
        <b>{turn}</b>&nbsp;|&nbsp;{state.mode}&nbsp;/&nbsp;{state.map_name}&nbsp;|&nbsp;{p1p2}&nbsp;|&nbsp;pick&nbsp;#{state.pick_number}{bans_s}
      </div>
      <div style="background:#34495e;color:#ecf0f1;padding:5px 14px;font-size:0.85em;">
        My&nbsp;team:&nbsp;<b style="color:#2ecc71;">{my_s}</b>
        &nbsp;&nbsp;Opp:&nbsp;<b style="color:#e74c3c;">{opp_s}</b>
        &nbsp;&nbsp;skill_ns={state.skill_ns:.1f}&nbsp;&nbsp;root&nbsp;Q={root_q:.3f}
      </div>
      <div style="background:#ecf0f1;padding:5px 14px;font-size:0.82em;border-bottom:1px solid #ddd;">
        <span style="color:{lc};font-weight:bold;">{label}</span>
        &nbsp;|&nbsp;{total:,}&nbsp;sims&nbsp;|&nbsp;{n_avail}&nbsp;choices
        &nbsp;|&nbsp;UCB1&nbsp;C={ucb1_c:.2f}
        &nbsp;|&nbsp;{result.elapsed_sec:.2f}s
        &nbsp;|&nbsp;Layer&nbsp;1:&nbsp;{l1.get('confidence_label', 'N/A')}
      </div>
      <table style="width:100%;border-collapse:collapse;background:white;">
        <thead>
          <tr style="background:#2c3e50;color:white;text-align:left;font-size:0.85em;">
            <th style="padding:6px 8px;width:24px;">#</th>
            <th style="padding:6px 8px;width:140px;">Brawler</th>
            <th style="padding:6px 8px;width:55px;text-align:right;">Visits</th>
            <th style="padding:6px 10px;">Visit %</th>
            <th style="padding:6px 8px;width:75px;text-align:right;">Win Prob</th>
            <th style="padding:6px 8px;width:65px;text-align:right;">Δ Win%</th>
            <th style="padding:6px 8px;width:55px;text-align:right;">vs Rnd</th>
          </tr>
        </thead>
        <tbody>
    """

    for i, p in enumerate(picks):
        b     = p["brawler"]
        vc    = p["visit_count"]
        vf    = p["visit_fraction"]
        wp    = p["estimated_win_prob"]
        delta = p["win_prob_delta"]
        rvf   = p["relative_visit_fraction"]

        row_bg = "#fffbf0" if i == 0 else ("#f9f9f9" if i % 2 == 0 else "white")
        dc     = "#27ae60" if delta >= 0 else "#e74c3c"
        fw     = "bold" if i == 0 else "normal"

        if vc < 30:
            flag = "&nbsp;<span title='Very few visits — unreliable' style='color:#e74c3c;'>⚠</span>"
        elif vc < 100:
            flag = "&nbsp;<span title='Low visits — treat with caution' style='color:#f39c12;'>⚡</span>"
        else:
            flag = ""

        html += f"""
          <tr style="background:{row_bg};">
            <td style="padding:5px 8px;color:#95a5a6;font-size:0.85em;">{i+1}</td>
            <td style="padding:5px 8px;font-weight:{fw};">{b}{flag}</td>
            <td style="padding:5px 8px;text-align:right;color:#7f8c8d;font-size:0.9em;">{vc:,}</td>
            <td style="padding:5px 10px;">{_visit_bar(vf)}&nbsp;<span style='font-size:0.85em;'>{vf*100:.1f}%</span></td>
            <td style="padding:5px 8px;text-align:right;">{wp:.3f}</td>
            <td style="padding:5px 8px;text-align:right;color:{dc};">{delta:+.3f}</td>
            <td style="padding:5px 8px;text-align:right;color:#95a5a6;font-size:0.9em;">{rvf:.1f}×</td>
          </tr>
        """

    html += """
        </tbody>
      </table>
      <div style="background:#f8f9fa;padding:4px 14px;font-size:0.78em;color:#95a5a6;border-radius:0 0 6px 6px;border-top:1px solid #ecf0f1;">
        Ranked by <b>visit count</b> (not win prob) — more robust at low sim counts &nbsp;·&nbsp;
        ⚠ &lt;30 visits (unreliable) &nbsp;·&nbsp; ⚡ &lt;100 visits (treat with caution) &nbsp;·&nbsp;
        Win Prob = mean FM win rate from rollouts &nbsp;·&nbsp;
        Δ Win% = vs current root Q &nbsp;·&nbsp;
        vs Rnd = vs uniform baseline
      </div>
    </div>
    """
    return html


# ── Button callbacks ──────────────────────────────────────────────────────────

def _run(_):
    with output:
        clear_output(wait=True)
        my_picks  = my_picker.picks
        opp_picks = opp_picker.picks
        bans      = ban_picker.picks
        map_name  = map_dropdown.value
        mode      = mode_dropdown.value
        skill_ns  = skill_slider.value
        is_fp     = p1p2_toggle.value

        # ── Complete draft: skip MCTS, evaluate directly ──────────────────
        if len(my_picks) == 3 and len(opp_picks) == 3:
            state = DraftState(
                my_team      = frozenset(b.upper() for b in my_picks),
                opp_team     = frozenset(b.upper() for b in opp_picks),
                mode         = mode,
                map_name     = map_name,
                skill_ns     = skill_ns,
                is_first_pick= is_fp,
                bans         = frozenset(b.upper() for b in bans),
            )
            win_prob = evaluator.evaluate(state)
            display(HTML(_format_complete_draft(state, win_prob)))
            return

        # ── Partial draft: run MCTS ────────────────────────────────────────
        n_sims  = sims_slider.value
        ucb1_c  = ucb1c_slider.value
        n_top   = ntop_slider.value
        mpr     = min_pr_slider.value
        puct_a  = puct_slider.value
        pol     = policy if use_policy_chk.value else None
        seed    = seed_input.value

        print(f"Running {n_sims:,} sims  |  UCB1 C={ucb1_c:.2f}  |  top {n_top}  |  seed={seed}…")
        try:
            result = recommend(
                my_picks      = my_picks,
                opp_picks     = opp_picks,
                mode          = mode,
                map_name      = map_name,
                skill_ns      = skill_ns,
                is_first_pick = is_fp,
                bans          = bans,
                n_simulations = n_sims,
                n_top         = n_top,
                ucb1_c        = ucb1_c,
                puct_alpha    = puct_a,
                min_pick_rate = mpr,
                evaluator     = evaluator,
                db            = db,
                rng           = np.random.default_rng(seed),
                policy        = pol,
            )
        except ValueError as e:
            print(f"\nError: {e}")
            return

        display(HTML(_format_result(result, ucb1_c=ucb1_c)))


def _clear(_):
    with output:
        clear_output()


def _reset(_):
    my_picker.clear()
    opp_picker.clear()
    ban_picker.clear()
    with output:
        clear_output()


run_btn.on_click(_run)
clear_btn.on_click(_clear)
reset_btn.on_click(_reset)


# ── Full layout ───────────────────────────────────────────────────────────────

sep = lambda: w.HTML("<hr style='margin:10px 0;border:none;border-top:1px solid #ecf0f1;'>")

draft_col = w.VBox([
    w.HTML("<h3 style='margin:0 0 10px 0;color:#2c3e50;'>Draft State</h3>"),
    w.HBox([w.HTML("<b>Map:</b>&nbsp;"), map_dropdown]),
    w.HBox([w.HTML("<b>Mode:</b>&nbsp;"), mode_dropdown]),
    p1p2_toggle,
    skill_slider,
    skill_note,
    sep(),
    w.HBox([my_picker.widget, opp_picker.widget]),
    sep(),
    ban_picker.widget,
], layout=w.Layout(min_width="440px", margin="0 24px 0 0"))

param_col = w.VBox([
    w.HTML("<h3 style='margin:0 0 10px 0;color:#2c3e50;'>MCTS Parameters</h3>"),
    sims_slider,
    ucb1c_slider,
    ucb1c_note,
    sep(),
    ntop_slider,
    min_pr_slider,
    min_pr_note,
    puct_slider,
    puct_note,
    sep(),
    use_policy_chk,
    seed_input,
    sep(),
    w.HBox([run_btn, w.HTML("&nbsp;"), clear_btn, w.HTML("&nbsp;"), reset_btn]),
], layout=w.Layout(min_width="420px"))

ui = w.VBox([
    w.HTML(
        "<div style='background:#2c3e50;color:white;padding:10px 16px;border-radius:6px;margin-bottom:16px;'>"
        "<h2 style='margin:0;'>Draft Recommendation Playground</h2>"
        "<span style='font-size:0.85em;color:#bdc3c7;'>MCTS draft agent · Brawl Stars</span>"
        "</div>"
    ),
    w.HBox([draft_col, param_col]),
    output,
])

display(ui)


---
## Quick script-style calls

For fast iteration without the widget — useful for comparing different parameter settings side by side.

In [ ]:
# Quick comparison: exploit (C=0.5) vs explore (C=3.0)
# Edit these parameters and re-run the cell

QUICK_PARAMS = dict(
    my_picks      = [],
    opp_picks     = [],
    mode          = "gemGrab",
    map_name      = "Double Swoosh",
    skill_ns      = 2.0,
    is_first_pick = True,
    bans          = [],           # e.g. ["SIRIUS", "BULL"]
    n_simulations = 10_000,
    n_top         = 10,
    min_pick_rate = 0.005,
    evaluator     = evaluator,
    db            = db,
    policy        = policy,
)

for label, c in [("Exploit  (C=0.50)", 0.50), ("Balanced (C=2.00)", 2.00), ("Explore  (C=5.00)", 5.00)]:
    r = recommend(**QUICK_PARAMS, ucb1_c=c, rng=np.random.default_rng(42))
    print(f"\n── {label} ── {r.elapsed_sec:.1f}s")
    print(f"  {'Rank':<4} {'Brawler':<16} {'Visits':>7} {'Visit%':>7} {'WinProb':>8} {'Δ':>7} {'vsRnd':>6}")
    print(f"  {'----':<4} {'-------':<16} {'------':>7} {'------':>7} {'-------':>8} {'---':>7} {'-----':>6}")
    for i, p in enumerate(r.top_picks):
        flag = "⚠" if p["visit_count"] < 30 else ("⚡" if p["visit_count"] < 100 else " ")
        print(
            f"  {i+1:<4} {p['brawler']:<16} {p['visit_count']:>7,}"
            f"  {p['visit_fraction']*100:>5.1f}%"
            f"  {p['estimated_win_prob']:>8.3f}"
            f"  {p['win_prob_delta']:>+7.3f}"
            f"  {p['relative_visit_fraction']:>5.1f}×  {flag}"
        )
